# Ch6 (A, inline) - Generating Data with a Designed Experiment

The store from Ch4 is an *oracle*: every fact it holds is a known answer. We mine it for questions, enrich each across a DoE of presentation factors, and emit one corpus that trains the encoders (Ch7) and tests the RAG (Ch15). This notebook shows the suite API inline.

In [ ]:
import os, sys
KNOWLYTIX_SRC = os.environ.get("KNOWLYTIX_SRC", "/home/user/jupyterlab/GMS-knowlytix")
sys.path.insert(0, KNOWLYTIX_SRC)
REPO = os.path.join(os.path.dirname(os.getcwd()), "code") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "scripts"))   # project modules

In [ ]:
import torch
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.harness.suite import (
    Catalog, resolve, CatalogBaseSource, compose, graphdoe_design)

STORE = os.path.join(REPO, "data", "gms_annual_report_store")
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
store = GMSExpertStore(DocGMSConfig(store_path=STORE), device=dev)
assert store.load(), "build the store first (Ch4)"

## Mine base questions (content base types from the suite catalog)

In [ ]:
CAT = Catalog.load()
suite = resolve(CAT, ["exact_recall", "multi_hop"],
                ["clarity", "style", "length", "expertise", "paraphrase_depth"],
                mode="embedded")
items = CatalogBaseSource(store, max_per_category=8, seed=42).items(suite)
print(len(items), "base questions, each graph-derived")

## Design the experiment (embedded mode: #scenarios == n_runs)

In [ ]:
from functools import partial
scns = compose(suite, items, n_runs=150, seed=42,
               design_fn=partial(graphdoe_design, method="sobol+refine"),
               balance_base=True)
print(len(scns), "scenarios over", suite.factor_names)

## Materialize with Qwen, then emit the corpus

Factor levels become a natural question via a batched, guarded Qwen rewrite (full body in `scripts/enrich_data.py`). It emits the cohort, the v-space SFT rows, the u-space groups and the draft pairs; here we inspect the result.

In [ ]:
import json
cohort = json.load(open(os.path.join(REPO, "data", "enrichment", "rag_cohort.json")))
print(len(cohort), "cohort cases; sample:", cohort[0]["question"][:70])

**Self-check** - every case has a graph-derived answer and balanced factors.

In [ ]:
import collections
assert all(c["expected_answer"] is not None for c in cohort)
clar = collections.Counter(c["_factors"]["clarity"] for c in cohort)
assert min(clar.values()) >= 30
print("OK: designed, ground-truthed, balanced")